In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/metadata.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv


# 7. MODEL 1 — Ridge Regression (linear baseline)

In [16]:
# ==============================================================================
# 7. MODEL 1 — Ridge Regression (linear baseline)
# ==============================================================================
ridge_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", Ridge(alpha=5.0, random_state=RANDOM_STATE)),
])
ridge_pipe = evaluate_model("Ridge (baseline)", ridge_pipe)


Ridge (baseline)       | RMSLE=0.5280 | MAE=15,986.7 | R2=0.3779


**Insight — Ridge: RMSLE 0.5280, MAE $15,987, R² 0.378**

This is the weakest model by a wide margin — its RMSLE is more than double every tree-based model's. That's expected and useful: it establishes that a purely linear combination of ordinal-encoded features explains only ~38% of price variance, confirming what Section 2.5's scatter plots suggested — the real structure in this data is non-linear and driven by feature interactions (e.g. "how much a given `OperationalHoursMeter` matters *depends on* `Spec_BaseClass`"), which is exactly what the tree ensembles below are designed to exploit.


# 8. MODEL 2 — Random Forest Regressor

In [17]:
# ==============================================================================
# 8. MODEL 2 — Random Forest Regressor
# ==============================================================================
rf_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300, max_depth=16, min_samples_leaf=2,
        n_jobs=-1, random_state=RANDOM_STATE,
    )),
])
rf_pipe = evaluate_model("Random Forest", rf_pipe)


Random Forest          | RMSLE=0.2249 | MAE=6,159.0 | R2=0.8872


**Insight — Random Forest: RMSLE 0.2249, MAE $6,159, R² 0.887**

A massive jump over Ridge (0.528 → 0.225), confirming the non-linearity hypothesis directly. Random Forest is also the weakest of the four tree-based models here, which is a fairly typical pattern: bagged trees average out noise well but don't correct their own errors sequentially the way boosted trees do — that gap is what the gradient-boosting models below close further.


# 9. MODEL 3 — XGBoost Regressor

In [18]:
# ==============================================================================
# 9. MODEL 3 — XGBoost Regressor
# ==============================================================================
xgb_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", XGBRegressor(
        n_estimators=600, max_depth=7, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_lambda=1.0, random_state=RANDOM_STATE,
        tree_method="hist", n_jobs=-1,
    )),
])
xgb_pipe = evaluate_model("XGBoost", xgb_pipe)


XGBoost                | RMSLE=0.2137 | MAE=5,789.1 | R2=0.8981


**Insight — XGBoost: RMSLE 0.2137, MAE $5,789, R² 0.898 — the best untuned model**

XGBoost edges out LightGBM (0.2137 vs 0.2169) and beats Random Forest by a clear margin, which is why it's the model carried forward into hyperparameter tuning in Section 13. `tree_method="hist"` is what makes 600 trees at depth 7 practical on 111k rows within a normal notebook runtime rather than a multi-hour one.


# 10. MODEL 4 — LightGBM Regressor

In [19]:
# ==============================================================================
# 10. MODEL 4 — LightGBM Regressor
# ==============================================================================
lgbm_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LGBMRegressor(
        n_estimators=800, num_leaves=63, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1,
    )),
])
lgbm_pipe = evaluate_model("LightGBM", lgbm_pipe)


LightGBM               | RMSLE=0.2169 | MAE=5,918.9 | R2=0.8950


**Insight — LightGBM: RMSLE 0.2169, MAE $5,919, R² 0.895 — close second**

Within 1.5% RMSLE of XGBoost despite double the trees at a lower learning rate — LightGBM's leaf-wise growth is converging to a similar solution via a different path. The two being this close suggests the ceiling for boosted trees on these particular 55 features is somewhere around RMSLE 0.21, which matters for expectations: closing the remaining gap to the 0.20 cutoff likely needs *better features* (per the Section 2.3 and 2.8 notes on parsing the `colN` fields), not just a different boosting library.


# 11. MODEL 5 — CatBoost Regressor (native categorical handling)

In [20]:
# ==============================================================================
# 11. MODEL 5 — CatBoost Regressor (native categorical handling)
# ==============================================================================
catboost_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", CatBoostRegressor(
        iterations=800, depth=8, learning_rate=0.04,
        loss_function="RMSE", random_seed=RANDOM_STATE, verbose=False,
    )),
])
catboost_pipe = evaluate_model("CatBoost", catboost_pipe)


CatBoost               | RMSLE=0.2278 | MAE=6,243.3 | R2=0.8842


**Insight — CatBoost: RMSLE 0.2278, MAE $6,243, R² 0.884 — weakest of the four tree models**

A little surprising at first glance, since CatBoost is usually the strongest performer on datasets this categorical-heavy — but here it's fed the *already ordinal-encoded* columns rather than raw categories, which forfeits the ordered target-statistics encoding that's normally its main advantage. Worth flagging as a concrete next experiment: passing CatBoost the original (unencoded) categorical columns via its `cat_features` parameter, bypassing `OrdinalEncoder` for this model specifically, would likely close some of this gap.
